# Hyperparameter Tuning with Optuna
# BCSS Breast Cancer Semantic Segmentation

Tunes training HPs using ResNet34 U-Net (pretrained) as reference model.
Best HPs (lr, batch_size, weight_decay, optimizer, loss_fn) will be used for all 3 model variants.

**Enable GPU before running.**

In [ ]:
!python -c "import torch; torch.randn(2,2,device='cuda')" 2>/dev/null || pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q optuna albumentations

In [ ]:
import sys, os, glob

# Auto-detect environment and find scripts + data
def find_file(root, filename):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return dirpath
    return None

def find_dir(root, dirname):
    for dirpath, dirnames, _ in os.walk(root):
        if dirname in dirnames:
            return dirpath
    return None

if os.path.exists('/kaggle/input'):
    ENV = 'kaggle'
    SCRIPTS_DIR = find_file('/kaggle/input', 'config.py')
    DATA_DIR = find_dir('/kaggle/input', 'train')
    OUTPUT_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    ENV = 'colab'
    SCRIPTS_DIR = find_file('/content', 'config.py')
    DATA_DIR = find_dir('/content', 'train')
    OUTPUT_DIR = '/content/outputs'
else:
    ENV = 'local'
    SCRIPTS_DIR = os.path.abspath('../src')
    DATA_DIR = os.path.abspath('../data/prepared')
    OUTPUT_DIR = os.path.abspath('../outputs')

assert SCRIPTS_DIR, 'Could not find scripts (config.py)'
assert DATA_DIR, 'Could not find data (train/ folder)'
sys.path.insert(0, SCRIPTS_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Environment: {ENV}')
print(f'Scripts: {SCRIPTS_DIR}')
print(f'Data: {DATA_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Scripts found: {sorted(f for f in os.listdir(SCRIPTS_DIR) if f.endswith(".py"))}')
print(f'Data splits: {sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))}')

In [ ]:
import torch
from train_utils import detect_amp_dtype

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

x = torch.randn(4, 4, device="cuda")
print(f"CUDA test: OK")

AMP_DTYPE = detect_amp_dtype()
print(f"AMP dtype: {AMP_DTYPE}")

In [ ]:
import json
import optuna
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

from config import NUM_CLASSES
from dataset import BCSSDataset, get_train_transform, get_valid_transform, compute_class_weights
from models import build_model
from train_utils import train_one_epoch, evaluate, build_criterion

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class_weights = compute_class_weights(DATA_DIR, split='train')
print(f'Class weights: {class_weights}')

## Define Optuna Objective

Fixed model: ResNet34 U-Net (pretrained). Tuning: lr, batch_size, weight_decay, optimizer, loss function.

In [ ]:
TUNING_EPOCHS = 10
NUM_WORKERS = 2

def objective(trial):
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])
    loss_fn = trial.suggest_categorical('loss_fn', ['ce', 'ce_dice'])

    train_ds = BCSSDataset(DATA_DIR, 'train', transform=get_train_transform())
    valid_ds = BCSSDataset(DATA_DIR, 'valid', transform=get_valid_transform())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model('resnet34', pretrained=True).to(device)
    criterion = build_criterion(loss_fn, class_weights=class_weights, device=device)

    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_dice = 0.0
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch(model, train_loader, criterion, optimizer, device, amp_dtype=AMP_DTYPE)
        val_metrics = evaluate(model, valid_loader, criterion, device)

        trial.report(val_metrics['mean_dice'], epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_metrics['mean_dice'] > best_dice:
            best_dice = val_metrics['mean_dice']

        print(f'  Trial {trial.number} | Epoch {epoch}/{TUNING_EPOCHS} | '
              f'mDice: {val_metrics["mean_dice"]:.4f} | best: {best_dice:.4f}')

    return best_dice

## Run Optuna Study

In [ ]:
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
    study_name='bcss_unet_hp_search',
)

study.optimize(objective, n_trials=25, show_progress_bar=True)

## Results

In [ ]:
print('=' * 60)
print(f'Best trial: #{study.best_trial.number}')
print(f'Best mean Dice: {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print('=' * 60)

best_params = study.best_params.copy()
best_params['best_dice'] = study.best_value

with open(os.path.join(OUTPUT_DIR, 'best_params.json'), 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'\nSaved to {OUTPUT_DIR}/best_params.json')

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances

fig1 = plot_optimization_history(study)
fig1.show()

fig2 = plot_param_importances(study)
fig2.show()

In [ ]:
import pandas as pd

trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values('value', ascending=False)
print(f'Completed: {len(trials_df[trials_df["state"] == "COMPLETE"])}')
print(f'Pruned: {len(trials_df[trials_df["state"] == "PRUNED"])}')
trials_df.head(10)